> **Bhavana Padala (25111063) - EXTRACT**: this notebook has only my section. In the final `01_ETL_Pipeline.ipynb`, it is pasted in order after the earlier sections, and everything runs top to bottom.

# PerfectPetClub – Customer Churn ETL Pipeline

**Module:** Data Engineering Team Presentation &nbsp;|&nbsp; **Notebook 1 of 2:** ETL pipeline

**Goal:** build a clean, one-row-per-customer dataset for analysing **customer churn**, where churn means a customer has **cancelled the auto-renew** of their subscription (`subscription.auto_renew == 0`).

| Section | Owner | What it does |
|---|---|---|
| 0. Setup & configuration | Team | Imports, paths, logging |
| 1. Extract | Bhavana Padala | Read SQLite DB, CRM CSV, server log CSV, survey CSV + question metadata |
| 2. Transform A – data quality & cleaning | Siddu Gundeboina | Data quality checks, type conversion, parsing, cleaning |
| 3. Transform B – feature engineering | Pravalika | Build customer-level features and the churn target |
| 4. Load | Pravalika | Write and verify the Parquet files |
| 5. Testing & validation | Denya Sree Adapa | Unit tests and end-to-end checks on the output |
| Repo, README, merge, HTML export | Ramya Sree Kasula & Denya Sree Adapa | Git repository, merging branches, run-all + HTML export |

**Architecture**

```
 SQLite DB ─┐
 CRM CSV  ──┤   extract_*()  ──►  raw dict  ──►  run_quality_checks() ──► clean_*()  ──►  build_*_features()
 Server log ┤                                   (DQ report)               (typed,         (one row per customer)
 Survey CSV ┘                                                              parsed)               │
                                                                                                  ▼
                                                               load_parquet()  ──►  output/customer_churn.parquet
                                                                                    output/customer_churn_model_ready.parquet
```

## 0. Setup & configuration

In [ ]:
import ast
import logging
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ---- Configuration (change paths here, not inside functions) ----
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCES = {
    "db": DATA_DIR / "PerfectPetClub.db",
    "crm": DATA_DIR / "PerfectPetClub_CRM_report_1.csv",
    "server_log": DATA_DIR / "PerfectPetClub_server_log_1.csv",
    "survey": DATA_DIR / "PerfectPetClub_survey.csv",
    "survey_questions": DATA_DIR / "PerfectPetClub_survey_questions.txt",
}
DB_TABLES = ["customer", "subscription", "subscription_type", "sale", "sale_product",
             "product", "product_category", "payment", "product_review"]

OUTPUT_FILE = OUTPUT_DIR / "customer_churn.parquet"
MODEL_READY_FILE = OUTPUT_DIR / "customer_churn_model_ready.parquet"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("ppc_etl")
log.info("pandas %s | numpy %s", pd.__version__, np.__version__)

## 1. Extract  *(owner: Bhavana Padala)*

Four sources, four different formats, so each one has its own small extract function. All of them:
- check the file exists first and raise a clear `FileNotFoundError` if not (fail fast, before any processing);
- return **raw** data with no cleaning, so the Transform stage owns every change made to the data;
- log the number of rows read, giving an audit trail for each run.

| Source | Format | Contents |
|---|---|---|
| `PerfectPetClub.db` | SQLite | customers, subscriptions, sales, products, payments, reviews |
| `CRM_report_1.csv` | CSV | customer-service contacts (complaints / queries) |
| `server_log_1.csv` | CSV (~67k rows) | web-app logins with device user-agent |
| `survey.csv` + questions `.txt` | CSV + text | pop-up survey answers and the question wording |

In [ ]:
def _check_exists(path: Path) -> Path:
    """Raise a helpful error if a source file is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Source file not found: {path.resolve()}")
    return path


def extract_sqlite(db_path: Path, tables: list[str]) -> dict[str, pd.DataFrame]:
    """Read each requested table from the SQLite database into a DataFrame."""
    _check_exists(db_path)
    frames = {}
    # 'with' closes the connection even if a query fails
    with sqlite3.connect(db_path) as conn:
        available = set(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"])
        missing = set(tables) - available
        if missing:
            raise ValueError(f"Tables missing from database: {sorted(missing)}")
        for table in tables:
            frames[table] = pd.read_sql(f"SELECT * FROM {table}", conn)
            log.info("Extracted %-17s %6d rows", table, len(frames[table]))
    return frames


def extract_csv(path: Path, name: str, **kwargs) -> pd.DataFrame:
    """Generic CSV reader with existence check and logging."""
    _check_exists(path)
    df = pd.read_csv(path, **kwargs)
    log.info("Extracted %-17s %6d rows, %d cols", name, len(df), df.shape[1])
    return df


def extract_survey_questions(path: Path) -> dict[str, str]:
    """Parse the survey questions file into {column_name: question text}."""
    _check_exists(path)
    questions = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if "=" in line:
            col, text = line.split("=", 1)
            questions[col.strip()] = text.strip()
    log.info("Extracted %d survey question definitions", len(questions))
    return questions


def extract_all(sources: dict = SOURCES) -> dict:
    """Run every extractor and return all raw data in a single dictionary."""
    raw = extract_sqlite(sources["db"], DB_TABLES)
    raw["crm"] = extract_csv(sources["crm"], "crm")
    raw["server_log"] = extract_csv(sources["server_log"], "server_log")
    raw["survey"] = extract_csv(sources["survey"], "survey")
    raw["survey_questions"] = extract_survey_questions(sources["survey_questions"])
    return raw

In [ ]:
raw = extract_all()

# Quick look at what we extracted
pd.DataFrame(
    {name: {"rows": len(df), "columns": df.shape[1]} for name, df in raw.items() if isinstance(df, pd.DataFrame)}
).T

In [ ]:
# Raw samples: note the issues the Transform stage must handle
display(raw["crm"].head(3))           # typo 'recieved_datatime', platform stored as a list-string "['e-mail']"
display(raw["server_log"].head(3))    # user_agent is truncated, device info is inside brackets
display(raw["survey"].head(3))        # income is a text band e.g. '26k-30k', '198k-200k+'
raw["survey_questions"]